In [1]:
import queue
import threading

from sqlalchemy import create_engine, text
from db import db_engine, get_recording, insert_recording
from pipeline import final_pipeline

recordings_queue = queue.Queue()
worker_running = False
worker_thread = None


def handle_recording(recording: dict):
    """Save the recording row right away. Queue it only if it should be processed now."""
    file_id, was_inserted = insert_recording(recording, status="received")

    if not was_inserted:
        print(f"[Duplicate] file already exists as file_id={file_id}; not queued")
        return file_id

    if recording.get("process_now") is True:
        recordings_queue.put(file_id)
        print(f"[Queued] file_id={file_id} | queue size: {recordings_queue.qsize()}")

    return file_id


def process_recording(file_id: int):
    """Process one recording by its file_id."""
    recording = get_recording(file_id)
    print(f"[Processing] file_id={file_id}:", recording)

    final_pipeline(recording)
    print(f"[Complete] file_id={file_id}")


def get_waiting_ids():
    """Return recording IDs still waiting in 'received' status."""
    with db_engine.connect() as conn:
        rows = conn.execute(
            text("SELECT file_id FROM recordings WHERE status = 'received' ORDER BY recorded_at ASC NULLS LAST, file_id ASC")
        ).fetchall()
    return [row[0] for row in rows]


def process_waiting():
    """Queue all recordings currently in 'received' status."""
    file_ids = get_waiting_ids()
    for file_id in file_ids:
        recordings_queue.put(file_id)
    print(f"[Catch-up Queued] {len(file_ids)} recording(s)")


def worker_loop():
    """Background worker taking recording IDs from the queue one at a time."""
    while worker_running:
        try:
            file_id = recordings_queue.get(timeout=0.5)
        except queue.Empty:
            continue

        try:
            process_recording(file_id)
        except Exception as error:
            print(f"[Worker Error] file_id={file_id}: {error}")
        finally:
            recordings_queue.task_done()


def start_pipeline():
    global worker_running, worker_thread
    if not worker_running:
        worker_running = True
        worker_thread = threading.Thread(target=worker_loop, name="WorkerThread", daemon=True)
        worker_thread.start()
        print("Worker started.")


def stop_pipeline():
    global worker_running
    worker_running = False
    print("Worker stopping.")


from recorder import start_keyboard_recorder

try:
    recorder.shutdown()
except NameError:
    pass

start_pipeline()
#process_waiting()

recorder = start_keyboard_recorder(
    on_recording_saved=handle_recording,
    on_shutdown=stop_pipeline,
)


Worker started.
Recorder ready: x toggles recording; 0 shuts down the recorder.
[Recorder] recording started
[Recorder] saved 20261128_132044_317969_c4dbfb4e.wav; process_now=True
[Queued] file_id=8816 | queue size: 1
[Processing] file_id=8816: {'file_id': 8816, 'file_path': 'C:\\Users\\DELL\\Desktop\\shop_recordings\\20261128_132044_317969_c4dbfb4e.wav', 'recorded_at': datetime.datetime(2026, 11, 28, 13, 20, 44, 317969, tzinfo=datetime.timezone(datetime.timedelta(seconds=18000))), 'status': 'received', 'issue': None, 'error_log': [], 'raw_transcript': None, 'extracted_json': None, 'processed_at': None}
[Complete] file_id=8816
Recorder stopped.
Worker stopping.


In [7]:
start_pipeline()
process_waiting()

[Catch-up Queued] 0 recording(s)


In [3]:
for i in range(1): 
    handle_recording({
        "recorded_at": "2026-11-24T12:00:00+05:00",
        "file_path": r"C:\Users\DELL\Desktop\shop_recordings\test407.m4a",
        "duration_seconds": 5.42,
        "process_now": True,
    })

[Duplicate] file already exists as file_id=8760; not queued
